In [24]:
import pacmap
import umap.umap_                       as umap 
import numpy                            as np
import pandas                           as pd

import seaborn                          as sns
import matplotlib.pyplot                as plt
import matplotlib.gridspec              as gridspec
import plotly.express                   as px
import plotly.graph_objects             as go

from tqdm                               import tqdm
from plotly.subplots                    import make_subplots
from sklearn.preprocessing              import StandardScaler, MinMaxScaler
from sklearn.cluster                    import DBSCAN, KMeans 
from sklearn.mixture                    import GaussianMixture
from sklearn.decomposition              import PCA

from utils.Preprocessing2               import PreProcessData
from utils.plotFigures                  import lineAndScatterPlot, linePlot, scatterPlot2D, plot_learning_curves
from clustering.plotClustering          import ClusteringResults


In [25]:
preData = PreProcessData()
df, scaled_data, df_healthy, scaled_healthy = preData.preprocessing(type_file           = 'h5', 
                                                                    scaler              = MinMaxScaler(), 
                                                                    window_size         = 6, 
                                                                    time_aggregation    = '10min', 
                                                                    threshold           = 200)

# Manipulable

In [26]:
df_manipulable = scaled_data[preData.manipulated_cr1+preData.new_manipulable_cr1]

clusteringModels    = {'pacmap': GaussianMixture(n_components = 2, 
                                                 random_state = 2,  
                                                 n_init       = 5)} 

dimReductionModels  = {'pacmap': pacmap.PaCMAP(n_components = 2, 
                                               random_state = 40, 
                                               save_tree    = True)}

Manipulable = ClusteringResults(data                = scaled_data, 
                                dataNormalized      = df_manipulable, 
                                clusteringModels    = clusteringModels, 
                                dimReductionModels  = dimReductionModels)

# Train dimensionality reduction model
Manipulable.train()
Manipulable.getResults2()

# Calculate clusters for each trajectory
for traj in tqdm(Manipulable.trajectories):
    Manipulable.clustersResults[traj] = Manipulable.embedding[Manipulable.data['trajectory'] == traj]

c:\Users\CAEX\Desktop\Diego\Crusher Project\2ndrealdatajournalchile\mmodal\Lib\site-packages\pacmap\pacmap.py:828: UserWarning:


100%|██████████| 24/24 [00:00<00:00, 630.99it/s]


In [27]:
columns         = preData.efficiency_cr1+preData.new_efficiency_cr1
df_efficiency   = scaled_data[columns]


clusteringModels    = {'pacmap': GaussianMixture(n_components = 5, 
                                                 random_state = 3, 
                                                 n_init       = 5)} 

dimReductionModels  = {'pacmap': pacmap.PaCMAP(n_components = 2, 
                                               random_state = 40, 
                                               save_tree    = True)}

Efficiency = ClusteringResults(data                = scaled_data, 
                               dataNormalized      = df_efficiency, 
                               clusteringModels    = clusteringModels, 
                               dimReductionModels  = dimReductionModels)

# Train dimensionality reduction model
Efficiency.train()
Efficiency.getResults2()

# Calculate clusters for each trajectory
for traj in tqdm(Efficiency.trajectories):
    Efficiency.clustersResults[traj] = Efficiency.embedding[Efficiency.data['trajectory'] == traj]

c:\Users\CAEX\Desktop\Diego\Crusher Project\2ndrealdatajournalchile\mmodal\Lib\site-packages\pacmap\pacmap.py:828: UserWarning:


100%|██████████| 24/24 [00:00<00:00, 726.61it/s]


# Visualization

## Embedding colored by variables

In [64]:
scatterPlot2D(data  = Manipulable.embedding, 
              x     = f'pacmap_x', 
              y     = f'pacmap_y', 
              color = preData.new_manipulable_cr1[1])

In [29]:
scatterPlot2D(data  = Efficiency.embedding, 
              x     = f'pacmap_x', 
              y     = f'pacmap_y', 
              color = preData.efficiency_cr1[0])

## Clusters

In [30]:
scatterPlot2D(data  = Manipulable.embedding, 
              x     = f'pacmap_x', 
              y     = f'pacmap_y', 
              color = 'cluster_pacmap')

In [31]:
scatterPlot2D(data  = Efficiency.embedding, 
              x     = f'pacmap_x', 
              y     = f'pacmap_y', 
              color = 'cluster_pacmap')

## Time serie colored by cluster

In [32]:
variable = preData.new_manipulable_cr1[1]
i = 2
lineAndScatterPlot(Manipulable.embedding[Manipulable.embedding['tiempo'].between('2023-01-01 00:00:00', '2023-01-10 00:00:00')], x='tiempo', y=variable, color='cluster_pacmap')

In [33]:
variable = preData.efficiency_cr1[0]
i = 2
lineAndScatterPlot(Efficiency.embedding, x='tiempo', y=variable, color='cluster_pacmap')